# PySpark Homework - Split by Steps
This notebook breaks Task 1 and Task 2 into small, runnable sections for better understanding.

## 0. Setup: Imports and Spark Session

In [ ]:
from pathlib import Path
import random
import string
import warnings

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, StructField, StructType

# Prefer the new package name first; fallback to legacy package if needed.
try:
    from data_profiling import ProfileReport
    profiling_backend = "data_profiling"
except Exception:
    try:
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=DeprecationWarning)
            from ydata_profiling import ProfileReport
        profiling_backend = "ydata_profiling"
    except Exception:
        ProfileReport = None
        profiling_backend = None

spark = SparkSession.builder.appName("Task1_Task2_PySpark_Notebook").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# sales_path = "C:/Users/KunalMajumdar/OneDrive - EPAM/EPAM Trainings/Spark for DQE/sales.csv"
# sellers_path = "C:/Users/KunalMajumdar/OneDrive - EPAM/EPAM Trainings/Spark for DQE/sellers.csv"
sales_path = "sales.csv"
sellers_path = "sellers.csv"
output_dir = Path("generated_report")
output_dir.mkdir(parents=True, exist_ok=True)

print("Spark initialized")
print(f"sales_path: {sales_path}")
print(f"sellers_path: {sellers_path}")
print(f"output_dir: {output_dir.resolve()}")
print(f"profiling_backend: {profiling_backend}")

## Task 1.1: Load sales.csv and filter seller_id = 2

In [ ]:
sales_df = spark.read.csv(sales_path, header=True, inferSchema=True)
sales_df_seller_2 = sales_df.filter(F.col("seller_id") == 2)

print(f"sales_df count: {sales_df.count()}")
print(f"sales_df_seller_2 count: {sales_df_seller_2.count()}")
sales_df_seller_2.show(5, truncate=False)

## Task 1.2: Create ydata-profiling reports (HTML + JSON)

In [ ]:
if ProfileReport is None:
    print("Profiling library is not installed in this kernel.")
    print("Run this in a new cell if needed: %pip install fg-data-profiling")
    print("Task 1.2 skipped for now. Continue with the next cells.")
else:
    import json

    seller2_pdf = sales_df_seller_2.toPandas()

    html_report_path = output_dir / "sales_df_seller_2_profile.html"
    json_report_path = output_dir / "sales_df_seller_2_profile.json"

    report_written = False

    # Attempt 1: full explorative profiling.
    try:
        profile = ProfileReport(
            seller2_pdf,
            title="sales_df_seller_2 profiling",
            explorative=True,
            progress_bar=False,
        )
        profile.to_file(html_report_path.as_posix())
        json_report_path.write_text(profile.to_json(), encoding="utf-8")
        report_written = True
        print("Profiling completed in explorative mode.")
    except Exception as e1:
        print(f"Explorative mode failed: {e1}")

    # Attempt 2: minimal profiling.
    if not report_written:
        try:
            profile = ProfileReport(
                seller2_pdf,
                title="sales_df_seller_2 profiling (minimal)",
                minimal=True,
                explorative=False,
                progress_bar=False,
            )
            profile.to_file(html_report_path.as_posix())
            json_report_path.write_text(profile.to_json(), encoding="utf-8")
            report_written = True
            print("Profiling completed in minimal mode.")
        except Exception as e2:
            print(f"Minimal mode failed: {e2}")

    # Attempt 3: guaranteed fallback outputs (simple summary files) so notebook never blocks.
    if not report_written:
        fallback_summary = {
            "warning": "ydata/fg profiling failed due to wordcloud rendering issue.",
            "row_count": int(len(seller2_pdf)),
            "column_count": int(len(seller2_pdf.columns)),
            "columns": list(seller2_pdf.columns),
            "dtypes": {k: str(v) for k, v in seller2_pdf.dtypes.items()},
            "null_counts": {k: int(v) for k, v in seller2_pdf.isnull().sum().to_dict().items()},
            "describe": seller2_pdf.describe(include="all").fillna("").to_dict(),
        }

        # default=str converts date/timestamp/numpy scalar values to JSON-safe strings.
        json_report_path.write_text(
            json.dumps(fallback_summary, ensure_ascii=False, indent=2, default=str),
            encoding="utf-8",
        )

        # Simple HTML report from the same fallback summary.
        describe_html = seller2_pdf.describe(include="all").fillna("").to_html()
        html_content = f"""
        <html>
        <head><title>sales_df_seller_2 fallback profile</title></head>
        <body>
        <h1>sales_df_seller_2 fallback profile</h1>
        <p><strong>Note:</strong> ydata/fg profiling failed, so this fallback summary was generated.</p>
        <p><strong>Rows:</strong> {len(seller2_pdf)}</p>
        <p><strong>Columns:</strong> {len(seller2_pdf.columns)}</p>
        <h2>Column Types</h2>
        <pre>{seller2_pdf.dtypes.to_string()}</pre>
        <h2>Describe (all columns)</h2>
        {describe_html}
        </body>
        </html>
        """.strip()
        html_report_path.write_text(html_content, encoding="utf-8")
        print("Generated fallback HTML/JSON summary report.")

    print(f"HTML report: {html_report_path}")
    print(f"JSON report: {json_report_path}")

## Task 1.3: Mask num_pieces_sold by '---'

In [ ]:
sales_df_seller_2_masked = sales_df_seller_2.withColumn("num_pieces_sold", F.lit("---"))
sales_df_seller_2_masked.select("order_id", "num_pieces_sold").show(5, truncate=False)

## Task 1.4: Change date format and validate with regexp_extract + when

In [ ]:
regex_date = r"^\\d{2}\\.\\d{2}\\.\\d{4}$"

sales_df_seller_2_date_checked = (
    sales_df_seller_2_masked
    .withColumn("date", F.date_format(F.to_date(F.col("date")), "dd.MM.yyyy"))
    .withColumn("regex_extract_result", F.regexp_extract(F.col("date"), regex_date, 0))
    .withColumn(
        "is_match_regex",
        F.when(F.col("regex_extract_result") != "", F.lit(True)).otherwise(F.lit(False))
    )
    .drop("regex_extract_result")
)

sales_df_seller_2_date_checked.select("date", "is_match_regex").show(10, truncate=False)

## Task 1.5: Expected vs actual schema mismatches

In [ ]:
def build_not_matched_schema_df(expected_schema: StructType, actual_schema: StructType):
    expected_map = {f.name: (f.dataType.simpleString(), f.nullable) for f in expected_schema.fields}
    actual_map = {f.name: (f.dataType.simpleString(), f.nullable) for f in actual_schema.fields}

    all_columns = sorted(set(expected_map.keys()).union(actual_map.keys()))
    rows = []

    for col_name in all_columns:
        exp = expected_map.get(col_name)
        act = actual_map.get(col_name)

        if exp == act:
            continue

        if exp is None:
            rows.append((col_name, "missing_in_expected", None, act[0], None, str(act[1])))
        elif act is None:
            rows.append((col_name, "missing_in_actual", exp[0], None, str(exp[1]), None))
        else:
            rows.append((col_name, "type_or_nullable_mismatch", exp[0], act[0], str(exp[1]), str(act[1])))

    mismatch_schema = StructType([
        StructField("column_name", StringType(), False),
        StructField("mismatch_type", StringType(), False),
        StructField("expected_type", StringType(), True),
        StructField("actual_type", StringType(), True),
        StructField("expected_nullable", StringType(), True),
        StructField("actual_nullable", StringType(), True),
    ])

    return spark.createDataFrame(rows, mismatch_schema)

expected_schema = StructType(sales_df.schema.fields + [StructField("warehouseId", IntegerType(), True)])
sales_df_seller_2_with_department = sales_df_seller_2_date_checked.withColumn("department", F.lit("test"))

not_matched_schema_df = build_not_matched_schema_df(
    expected_schema=expected_schema,
    actual_schema=sales_df_seller_2_with_department.schema
)

not_matched_schema_df.show(truncate=False)

## Task 2.1: Load sellers.csv

In [ ]:
sellers_df = spark.read.csv(sellers_path, header=True, inferSchema=True)
sellers_df.show(5, truncate=False)

## Task 2.2: Update rows and count nulls by column

In [ ]:
seller_id_int = F.col("seller_id").cast("int")

sellers_updated_df = (
    sellers_df
    .withColumn(
        "seller_name",
        F.when(seller_id_int == 3, F.lit("n/a"))
         .when(seller_id_int == 6, F.lit(None).cast("string"))
         .when(seller_id_int == 7, F.lit("NULL"))
         .otherwise(F.col("seller_name"))
    )
    .withColumn(
        "daily_target",
        F.when(seller_id_int == 3, F.lit("Undefined"))
         .when(seller_id_int == 6, F.lit("None"))
         .when(seller_id_int == 7, F.lit("61878"))
         .otherwise(F.col("daily_target").cast("string"))
    )
)

null_count_exprs = [
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in sellers_updated_df.columns
]
null_counts_df = sellers_updated_df.agg(*null_count_exprs)

print("Updated rows for seller_id in (3, 6, 7):")
sellers_updated_df.filter(seller_id_int.isin([3, 6, 7])).show(truncate=False)
print("Null counts per column:")
null_counts_df.show(truncate=False)

## Task 2.3: Add SSN using UDF

In [ ]:
@F.udf(returnType=StringType())
def generate_random_ssn() -> str:
    digits = "".join(random.choices(string.digits, k=9))
    return f"{digits[:3]}-{digits[3:5]}-{digits[5:]}"

sellers_with_ssn_df = sellers_updated_df.withColumn("SSN", generate_random_ssn())
sellers_with_ssn_df.show(5, truncate=False)

## Task 2.4: Mask SSN and drop original SSN column

In [ ]:
@F.udf(returnType=StringType())
def mask_ssn(ssn: str):
    if ssn is None:
        return None
    if len(ssn) <= 2:
        return ssn
    return ssn[0] + ("*" * (len(ssn) - 2)) + ssn[-1]

sellers_masked_ssn_df = sellers_with_ssn_df.withColumn("masked_SSN", mask_ssn(F.col("SSN"))).drop("SSN")
sellers_masked_ssn_df.show(10, truncate=False)

## Optional Final Cell: Stop Spark Session

In [ ]:
# Run this only when you are done with all tasks
# spark.stop()